# ESP streamflow ensemble evaluation

Score Ensemble Streamflow Prediction (ESP) traces against gauge observations.

- **CRPS / CRPSS** vs observational climatology (same calendar day, other years)
- **Peak-flow CRPS / CRPSS** (optional): one score per forecast for the maximum discharge over the horizon, versus observed peaks in that same calendar window
- **Water-volume CRPS / CRPSS**: one score per forecast for the total volume over the paired days, versus observed volume on those same calendar days
- **Block-bootstrap intervals** on the initiation-month means of those CRPS and CRPSS scores. Circular resampling is the default
- **Peak-over-horizon** Brier score and Brier skill score at the `Yp` thresholds, plus a reliability diagram and an ROC curve for the same ensemble exceedance probabilities
- Results are aggregated by forecast-initiation month (`02-01`, `03-01`, …)

This notebook calls library functions; it is not the source of truth. CLI equivalent:
`python -m snowda.workflows.evaluate_esp_streamflow ../runs/examples/ross_evaluate_esp.yaml`

In [ ]:
from pathlib import Path

%load_ext autoreload
%autoreload 2

## Decision points

These arguments control the evaluation. Defaults match the plan: fair CRPS, leave-one-out climatology, peak-over-horizon events, and a 0.5 probability threshold for POD/FAR. Set `SCORE_PEAK` to score the maximum discharge over each forecast horizon. `SCORE_VOLUME` scores total water volume over the paired days. The bootstrap settings put a circular block-bootstrap interval on the initiation-month mean CRPS and CRPSS.

In [ ]:
from pathlib import Path

RUN_DIR = Path("/Users/dcasson/GitHub/snowda/outputs/ross_summa_esp/20261027T144509Z")
OBS_CSV = Path("/Users/dcasson/GitHub/snowda/data/ross_daily_discharge.csv")
THRESHOLDS_CSV = Path("/Users/dcasson/Data/snowda/summa/ross_lumped/thresholds/ross_thresholds.csv")
OUTPUT_DIR = Path("/Users/dcasson/GitHub/snowda/outputs/ross_esp_streamflow_20_year_eval_da")

RETURN_PERIODS = (2, 5, 10, 20)
EVENT_PROBABILITY_THRESHOLD = 0.5
EXCLUDE_VERIFICATION_YEAR = True
CRPS_METHOD = "fair"  # "fair" or "ecdf"
FAIR_BRIER = True
SCORE_PEAK = True  # one peak-flow CRPS and CRPSS per forecast
SCORE_VOLUME = True  # one water-volume CRPS and CRPSS per forecast
BOOTSTRAP_ITERATIONS = 100  # 0 skips the confidence intervals
BOOTSTRAP_BLOCK_SIZE = 2
BOOTSTRAP_CIRCULAR = True
BOOTSTRAP_CI_LEVEL = 0.9
BOOTSTRAP_SEED = 0
RESAMPLE = "1D"
VARIABLE = "IRFroutedRunoff"
SEG = 0

## Load traces and score

In [ ]:
import pandas as pd

from snowda.io.esp_streamflow import load_esp_forecasts, read_return_period_thresholds
from snowda.workflows.evaluate_esp_streamflow import (
    compute_esp_metrics,
    esp_metrics_to_dict,
    write_esp_evaluation_products,
)

forecasts, obs = load_esp_forecasts(
    RUN_DIR,
    OBS_CSV,
    variable=VARIABLE,
    resample=RESAMPLE,
    seg=SEG,
)
thresholds = read_return_period_thresholds(THRESHOLDS_CSV)
metrics = compute_esp_metrics(
    forecasts,
    obs,
    thresholds,
    return_periods=RETURN_PERIODS,
    event_probability_threshold=EVENT_PROBABILITY_THRESHOLD,
    exclude_verification_year=EXCLUDE_VERIFICATION_YEAR,
    crps_method=CRPS_METHOD,
    fair_brier=FAIR_BRIER,
    score_peak=SCORE_PEAK,
    score_volume=SCORE_VOLUME,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    bootstrap_block_size=BOOTSTRAP_BLOCK_SIZE,
    bootstrap_circular=BOOTSTRAP_CIRCULAR,
    bootstrap_ci_level=BOOTSTRAP_CI_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)
payload = esp_metrics_to_dict(metrics)
print(f"Forecasts: {len(forecasts)}; init months: {payload['init_months']}")
metrics

In [ ]:
rows = []
for month, row in payload["by_init_month"].items():
    entry = {
        "init_month": month,
        "n_forecasts": row["n_forecasts"],
        "n_members": row["n_members"],
        "n_pairs": row["n_pairs"],
        "crps_mean": row["crps_mean"],
        "crpss_mean": row["crpss_mean"],
    }
    if "peak_crps_mean" in row:
        entry["peak_crps_mean"] = row["peak_crps_mean"]
        entry["peak_crpss_mean"] = row["peak_crpss_mean"]
    if "volume_crps_mean" in row:
        entry["volume_crps_mean"] = row["volume_crps_mean"]
        entry["volume_crpss_mean"] = row["volume_crpss_mean"]
    for name in (
        "crps_mean",
        "crpss_mean",
        "peak_crps_mean",
        "peak_crpss_mean",
        "volume_crps_mean",
        "volume_crpss_mean",
    ):
        for bound in ("ci_lower", "ci_upper"):
            key = f"{name}_{bound}"
            if key in row:
                entry[key] = row[key]
    for period in payload["return_periods"]:
        key = str(period)
        entry[f"bs_{key}"] = row["bs"][key]
        entry[f"bss_{key}"] = row["bss"][key]
        entry[f"pod_{key}"] = row["pod"][key]
        entry[f"far_{key}"] = row["far"][key]
    rows.append(entry)

summary = pd.DataFrame(rows).set_index("init_month")

peak_rows = []
for month, row in payload["by_init_month"].items():
    for init, crps in row.get("peak_crps", {}).items():
        peak_rows.append(
            {
                "init_time": init,
                "init_month": month,
                "peak_crps": crps,
                "peak_crpss": row["peak_crpss"][init],
            }
        )
volume_rows = []
for month, row in payload["by_init_month"].items():
    for init, crps in row.get("volume_crps", {}).items():
        volume_rows.append(
            {
                "init_time": init,
                "init_month": month,
                "volume_crps": crps,
                "volume_crpss": row["volume_crpss"][init],
            }
        )
if peak_rows or volume_rows:
    from IPython.display import display

    if peak_rows:
        display(pd.DataFrame(peak_rows).set_index("init_time"))
    if volume_rows:
        display(pd.DataFrame(volume_rows).set_index("init_time"))
summary

## Visualize

Hydrograph envelopes use the paired ESP traces. Metric plots use the aggregated `metrics` dataset. The lead-time plot shades the block-bootstrap interval when it was saved. The peak and volume plots show one CRPS and one CRPSS per initialization time; their intervals are on the month means in the summary table.

In [ ]:
from snowda.io.esp_streamflow import select_return_period_thresholds
from snowda.viz.esp_streamflow import (
    plot_crps_by_lead,
    plot_ensemble_envelopes,
    plot_ensemble_forecast_separate,
    plot_peak_scores,
    plot_threshold_metrics,
    plot_volume_scores,
)

plot_thresholds = select_return_period_thresholds(thresholds, RETURN_PERIODS)
PLOT_START = "2012-01-01"
PLOT_END = "2013-01-01"
fig, ax = plot_ensemble_envelopes(
    forecasts,
    obs,
    title="ESP ensembles vs observations",
    thresholds=plot_thresholds,
    start=PLOT_START,
    end=PLOT_END,
)

In [ ]:
from IPython.display import display

for forecast in forecasts:
    fig, ax = plot_ensemble_forecast_separate(
        forecast, obs, thresholds=plot_thresholds
    )

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
plot_crps_by_lead(metrics, ax=axes[0], skill=False)
plot_crps_by_lead(metrics, ax=axes[1], skill=True)
fig.tight_layout()

In [ ]:
fig = plot_peak_scores(metrics)
fig = plot_volume_scores(metrics)

In [ ]:
fig = plot_threshold_metrics(metrics)

## Reliability and ROC

Both diagrams use the peak-over-horizon ensemble probability: the fraction of members whose peak flow exceeds `Yp`, compared with whether the observed peak exceeds that same threshold. One panel per return period and one curve per initiation month. Empty reliability bins are omitted, and marker area shows how many forecasts fell in the bin.

In [ ]:
from snowda.adapters.scores import reliability_diagram, roc_diagram
from snowda.viz.esp_streamflow import plot_reliability_diagram, plot_roc_diagram

reliability = reliability_diagram(
    metrics["event_probability"],
    metrics["event_observed"],
)
roc = roc_diagram(
    metrics["event_probability"],
    metrics["event_observed"],
)
plot_reliability_diagram(reliability)
plot_roc_diagram(roc)

In [ ]:
from snowda.viz.esp_streamflow import plot_rank_histogram

plot_rank_histogram(metrics)

## Write artifacts (optional)

Writes `metrics.nc` and `metrics.json` into a timestamped folder under `OUTPUT_DIR`.

In [ ]:
from datetime import datetime, timezone

eval_dir = OUTPUT_DIR / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
write_esp_evaluation_products(metrics, eval_dir)
print("Wrote evaluation to", eval_dir)